# JPEG check

All real images in GenImage were JPEG before our preprocessing and all fakes PNG, so after the common JPEG q95
re-encoding (`data/preprocess.py`) reals have been compressed twice and fakes once. An expert could learn
"compressed before = real" instead of the generator's traces.

**Test:** give fakes a JPEG history too. Each fake is compressed once more (quality 90 or 75) and then re-encoded at
q95 as usual; the reals stay as they are. Each generator's own ResNet-50 expert scores them.

- **Share of fakes called fake** (logit > 0) drops a lot: the expert's decisions depend on compression.
- **AUROC stays high:** fakes still rank above reals, so compression is not the expert's only cue.

Extra compression also destroys some generator traces, so a drop alone can't separate a compression shortcut
from damaged artefacts. `test` is not looked at.

In [1]:
import io
import sys
sys.path.insert(0, "../..")  # repo root, so `data` and `resnet` are importable

import numpy as np
import pandas as pd
import torch
from PIL import Image
from sklearn.metrics import roc_auc_score

from common.device import get_device
from data import config
from data.images import load_manifest, open_image, select
from resnet.experts import EXPERTS
from resnet.model import ARCH, EVAL_TRANSFORM, load_expert

POOL = "meta_val"
N_IMAGES = 500  # per generator and label
QUALITIES = [90, 75]  # extra JPEG pass before the usual q95
BATCH_SIZE = 64

## Score reals, fakes and compressed fakes with the generator's own expert

In [2]:
def recompress(image, quality):
    """JPEG at quality, then the usual JPEG_QUALITY re-encoding of our preprocessing."""
    for q in (quality, config.JPEG_QUALITY):
        buffer = io.BytesIO()
        image.save(buffer, format="JPEG", quality=q)
        image = Image.open(io.BytesIO(buffer.getvalue())).convert("RGB")
    return image


def logits(model, images):
    out = []
    with torch.no_grad():
        for start in range(0, len(images), BATCH_SIZE):
            batch = torch.stack([EVAL_TRANSFORM(image) for image in images[start:start + BATCH_SIZE]]).to(device)
            out.append(model(batch).float().squeeze(1).cpu().numpy())
    return np.concatenate(out)


device = get_device()
manifest = load_manifest()
expert_of = {generator: expert for expert, generators in EXPERTS.items() for generator in generators}
rows = []
for generator in config.GENERATORS:
    model = load_expert(expert_of[generator], device, ARCH)[0]
    sample = lambda label: select(manifest, pool=POOL, generator=generator, label=label).sample(N_IMAGES, random_state=config.SEED)
    reals = logits(model, [open_image(file) for file in sample("real").file])
    fakes = [open_image(file) for file in sample("fake").file]
    row = {"generator": generator, "expert": expert_of[generator]}
    for quality, images in [("none", fakes)] + [(q, [recompress(image, q) for image in fakes]) for q in QUALITIES]:
        scores = logits(model, images)
        labels = np.r_[np.zeros(len(reals)), np.ones(len(scores))]
        row[f"fakes caught ({quality})"] = (scores > 0).mean()
        row[f"AUROC ({quality})"] = roc_auc_score(labels, np.r_[reals, scores])
    row["reals called fake"] = (reals > 0).mean()
    rows.append(row)

table = pd.DataFrame(rows).set_index("generator")
table.round(3)

,expert,fakes caught (none),AUROC (none),fakes caught (90),AUROC (90),fakes caught (75),AUROC (75),reals called fake
generator,,,,,,,,
ADM,ADM,0.982,0.998,0.616,0.950,0.316,0.916,0.028
BigGAN,BigGAN,1.000,1.000,0.882,0.998,0.450,0.984,0.002
glide,glide,0.994,1.000,0.652,0.981,0.232,0.931,0.002
Midjourney,Midjourney,0.968,0.997,0.918,0.990,0.746,0.972,0.024
stable_diffusion_v_1_4,SD,0.992,0.999,0.936,0.994,0.666,0.968,0.024
stable_diffusion_v_1_5,SD,0.992,1.000,0.882,0.994,0.648,0.971,0.008
VQDM,VQDM,0.986,0.998,0.794,0.983,0.682,0.977,0.010
wukong,wukong,0.990,0.999,0.938,0.993,0.686,0.964,0.016


## Summary

In [3]:
caught = [f"fakes caught ({q})" for q in ["none"] + QUALITIES]
auroc = [f"AUROC ({q})" for q in ["none"] + QUALITIES]
print("fakes caught, range over generators:", {c: f"{table[c].min():.1%}-{table[c].max():.1%}" for c in caught})
print("AUROC, lowest generator:", {c: round(table[c].min(), 3) for c in auroc})

fakes caught, range over generators: {'fakes caught (none)': '96.8%-100.0%', 'fakes caught (90)': '61.6%-93.8%', 'fakes caught (75)': '23.2%-74.6%'}
AUROC, lowest generator: {'AUROC (none)': np.float64(0.997), 'AUROC (90)': np.float64(0.95), 'AUROC (75)': np.float64(0.916)}
